In [ ]:
import csv
f = open("tanker_scheduling.csv")
csvfile = csv.DictReader(f, delimiter='\t')

headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

In [ ]:
print(headers)

In [ ]:
# read the parameters as floats
TravelTime = {}
Orders = {}
for row in table:
    TravelTime[row['Terminal']] = float(row['Travel Time'])
    # to read the orders, loop over 1 through 6 and only add non-empty order dates
    tmpOrders = set()
    for i in range(1,7):
        if row["Order "+str(i)] != "":
            tmpOrders.add(float(row["Order "+str(i)]))
    # add the orders to this terminal
    Orders[row['Terminal']] = tmpOrders

# be cautious: these dates are not ordered.  Check if this is needed.

In [ ]:
print(TravelTime)
print(Orders)

In [ ]:
# for clarity, define a set for the terminals (borrow key from TravelTime)
Terminals = { t for t in TravelTime }

# define the set of nodes: one for each terminal/order pair
# - we can enumerate these as numbers {1,2,...}
# - but then we need to define Terminal(i) and OrderDate(i) for each i in the set of nodes
Terminal = {}
OrderDate = {}
for t in Terminals:
    for o in Orders[t]:
        newNode = len(Terminal)+1
        Terminal[newNode] = t
        OrderDate[newNode] = o

Nodes = { i for i in Terminal }
        
# define arcs: (i,j) exists if delivery(i) + distance(i, Qatar) + distance(Qatar, j)  ≤ delivery(j)
Arcs = { (i,j) for i in Nodes for j in Nodes 
        if OrderDate[i] + TravelTime[Terminal[i]] + TravelTime[Terminal[j]] <= OrderDate[j] }

In [ ]:
# complete the network by adding 'source' and 'sink' nodes and associated arcs.
Nodes.add('source')
Nodes.add('sink')

In [ ]:
# connect each node to the source and to the sink
for i in Nodes:
    if i != 'source' and i != 'sink':
        Arcs.add(('source',i))
        Arcs.add((i,'sink'))

In [ ]:
# add one more arc from sink to source (representing the total flow through the network)
Arcs.add(('sink','source'))